# Compiler Spirula Studio sur Google Colab (T4)

Ce notebook installe les dépendances Vulkan, compile Spirula en Release avec le backend Vulkan et `SS_BUILD_SAM=ON`, télécharge le modèle SAM3 puis archive les résultats dans Google Drive.

**Avant de lancer :** dans `Runtime > Change runtime type`, sélectionner **T4 GPU** si disponible. Aucun numéro de carte bancaire n'est nécessaire pour exécuter ce notebook sur Colab gratuit, mais les GPU restent soumis aux quotas et disponibilités de Google.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Installation et compilation

L'archive finale sera écrite dans `Mon Drive/spirula-builds/`. La compilation se fait sur le disque local temporaire de Colab ; seul le résultat final est copié sur Drive.

In [ ]:
%%bash
set -Eeuo pipefail

REPO_URL="https://github.com/harry7557558/spirula-studio.git"
WORK_ROOT="/content/spirula-colab-build"
REPO_DIR="${WORK_ROOT}/spirula-studio"
DRIVE_OUT="/content/drive/MyDrive/spirula-builds"
TIMESTAMP="$(date +%Y%m%d_%H%M%S)"
LOG_FILE="/content/spirula_build_${TIMESTAMP}.log"
ARCHIVE="${DRIVE_OUT}/spirula-studio-vulkan-${TIMESTAMP}.zip"
SAM3_URL="https://huggingface.co/PABannier/sam3.cpp/resolve/main/sam3-q4_0.ggml"
SAM3_PATH="${REPO_DIR}/models/sam3-q4_0.ggml"

mkdir -p "${WORK_ROOT}" "${DRIVE_OUT}"
exec > >(tee -a "${LOG_FILE}") 2>&1
log() { echo "[Spirula-Colab] $*"; }
die() { log "[ERREUR] $*"; exit 1; }
trap 'status=$?; echo; log "Échec ligne ${LINENO} (code ${status}). Log: ${LOG_FILE}"; exit "${status}"' ERR

log "Vérification de la GPU"
command -v nvidia-smi >/dev/null 2>&1 || die "nvidia-smi absent : sélectionner un runtime GPU dans Colab."
nvidia-smi
GPU_NAME="$(nvidia-smi --query-gpu=name --format=csv,noheader | head -n1)"
log "GPU détectée : ${GPU_NAME}"
if [[ "${GPU_NAME}" != *"T4"* ]]; then
  log "AVERTISSEMENT : la GPU attribuée n'est pas une T4. Le notebook continue."
fi

log "Installation des dépendances système Vulkan / X11 / OpenGL / Wayland"
export DEBIAN_FRONTEND=noninteractive
apt-get update
apt-get install -y --no-install-recommends \
  build-essential pkg-config cmake ninja-build git curl wget ca-certificates zip unzip \
  libvulkan1 libvulkan-dev mesa-vulkan-drivers vulkan-tools \
  libx11-dev libxext-dev libxrender-dev libxinerama-dev libxcursor-dev libxi-dev libxrandr-dev libxfixes-dev \
  libwayland-dev libwayland-egl1 wayland-protocols libxkbcommon-dev \
  libgl1-mesa-dev libglx-dev libopengl-dev libegl-dev libgl-dev libglu1-mesa-dev
ldconfig

log "Contrôle des headers et bibliothèques Vulkan"
test -r /usr/include/vulkan/vulkan.h || die "Header /usr/include/vulkan/vulkan.h absent"
test -e /usr/lib/x86_64-linux-gnu/libvulkan.so || test -e /usr/lib64/libvulkan.so || die "libvulkan.so de développement absente"
pkg-config --exists vulkan || die "pkg-config vulkan indisponible"
log "Vulkan headers et pkg-config : OK ($(pkg-config --modversion vulkan))"
if command -v vulkaninfo >/dev/null 2>&1; then
  if vulkaninfo --summary; then
    log "Runtime Vulkan initialisable."
  else
    log "AVERTISSEMENT : vulkaninfo ne peut pas initialiser Vulkan dans ce runtime. La compilation peut néanmoins fonctionner."
  fi
fi

log "Clonage du dépôt Spirula Studio"
if [[ -d "${REPO_DIR}/.git" ]]; then
  git -C "${REPO_DIR}" pull --ff-only
else
  rm -rf "${REPO_DIR}"
  git clone "${REPO_URL}" "${REPO_DIR}"
fi
git -C "${REPO_DIR}" submodule sync --recursive
git -C "${REPO_DIR}" submodule update --init --recursive
test -f "${REPO_DIR}/build_develop.bash" || die "build_develop.bash introuvable"

log "Compilation Vulkan / Release / SAM"
cd "${REPO_DIR}"
JOBS="$(nproc)"
# Limite le parallélisme pour éviter les OOM sur les runtimes Colab.
if (( JOBS > 4 )); then JOBS=4; fi
export CMAKE_BUILD_PARALLEL_LEVEL="${JOBS}"
export CFLAGS="${CFLAGS:-} -I/usr/include"
export CXXFLAGS="${CXXFLAGS:-} -I/usr/include"
VULKAN_LIBRARY="$(ldconfig -p | awk '$1 == "libvulkan.so" {print $NF; exit}')"
if [[ -z "${VULKAN_LIBRARY}" ]]; then VULKAN_LIBRARY="/usr/lib/x86_64-linux-gnu/libvulkan.so"; fi
bash build_develop.bash \
  -DSS_BACKEND=vulkan \
  -DSS_BUILD_SAM=ON \
  -DCMAKE_BUILD_TYPE=Release \
  -DVulkan_INCLUDE_DIR:PATH=/usr/include \
  -DVulkan_LIBRARY:FILEPATH="${VULKAN_LIBRARY}"

BUILD_DIR="${REPO_DIR}/build_vulkan"
SPIRULA_BIN="${BUILD_DIR}/spirula"
test -x "${SPIRULA_BIN}" || die "Exécutable compilé absent ou non exécutable : ${SPIRULA_BIN}"
log "Exécutable compilé : ${SPIRULA_BIN}"
"${SPIRULA_BIN}" --version

log "Téléchargement du modèle SAM3"
mkdir -p "$(dirname "${SAM3_PATH}")"
if [[ ! -s "${SAM3_PATH}" ]]; then
  curl -fL --retry 3 --retry-delay 2 "${SAM3_URL}" -o "${SAM3_PATH}"
fi
test -s "${SAM3_PATH}" || die "Modèle SAM3 absent ou vide"
log "Modèle SAM3 : $(du -h "${SAM3_PATH}" | awk '{print $1}')"

log "Préparation des métadonnées et de l'archive"
PACKAGE_ROOT="${WORK_ROOT}/package"
rm -rf "${PACKAGE_ROOT}"
mkdir -p "${PACKAGE_ROOT}/spirula-studio"
cp -a "${BUILD_DIR}" "${PACKAGE_ROOT}/spirula-studio/"
mkdir -p "${PACKAGE_ROOT}/spirula-studio/models"
cp -a "${SAM3_PATH}" "${PACKAGE_ROOT}/spirula-studio/models/"
cp "${LOG_FILE}" "${PACKAGE_ROOT}/build.log"
git -C "${REPO_DIR}" rev-parse HEAD > "${PACKAGE_ROOT}/git_commit.txt"
cat > "${PACKAGE_ROOT}/README.txt" <<EOF
Spirula Studio - build Linux Vulkan Release pour Google Colab
GPU de compilation : ${GPU_NAME}
Commit : $(git -C "${REPO_DIR}" rev-parse HEAD)
Binaire : spirula-studio/build_vulkan/spirula
Modèle : spirula-studio/models/sam3-q4_0.ggml

Attention : cette archive contient le résultat de compilation et le modèle SAM3.
Le modèle SAM3 est soumis aux conditions de licence de son éditeur (Meta / dépôt de distribution).
La présence du binaire ne garantit pas que le runtime Vulkan sera disponible sur une autre machine.
EOF
cd "${PACKAGE_ROOT}"
zip -qr "${ARCHIVE}" .
log "Archive créée : ${ARCHIVE}"
ls -lh "${ARCHIVE}"
log "Terminé. Le ZIP est dans Mon Drive/spirula-builds/"


[Spirula-Colab] Vérification de la GPU
Sat Nov 14 20:32:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------

In [ ]:
!LOG=$(ls -t /content/spirula_build_*.log | head -n1)
!echo "Suivi du fichier : $LOG"
!tail -n 40 -f "$LOG"

Suivi du fichier : 
tail: cannot open '' for reading: No such file or directory
tail: no files remaining


## Résultat

À la fin, le ZIP se trouve dans `Mon Drive/spirula-builds/`. Le dossier de build est `spirula-studio/build_vulkan/` et le binaire `spirula-studio/build_vulkan/spirula`.

**Note Vulkan :** Colab peut exposer la GPU NVIDIA pour CUDA sans exposer un ICD Vulkan NVIDIA utilisable. Le notebook distingue donc les dépendances nécessaires à la compilation du test du runtime Vulkan. Si `vulkaninfo` échoue, le build peut quand même réussir, mais il faudra vérifier l'exécution Vulkan sur la machine cible.
